In [1]:
# notebook : semdetect_claims_local.ipynb — cellule 0 : imports et configuration
import json
import logging
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI

from src.data.dataset_semdetect import SemDetectDataset
from src.detector.sem_detect.claims_llm import LocalClaimExtractor
from src.detector.sem_detect.feature_store import parse_claims
from src.detector.sem_detect.semdetect.claim_extractor import CLAIM_CATEGORIES, SEMANTIC_CATEGORIES

logging.getLogger("httpx").setLevel(logging.WARNING)

CACHE_ROOT = Path(os.environ["CACHE_DIR"])
VLLM_URL = "http://127.0.0.1:8000/v1"
REVISION = "18b74b551670bff6b04e57570232a1694997863b"
SAMPLE_PER_CLASS = {"human": 34, "rewrite": 33, "ai": 33}  # 100 reviews
SEED = 0

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-03 10:43:49.402 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


In [2]:
# notebook : semdetect_claims_local.ipynb — cellule 1 : test Sem-Detect et échantillon stratifié
te = SemDetectDataset(config={
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews", "revision": REVISION,
    "split": "test", "conference": None, "date": None, "generator": None,
    "label_mode": "three_class", "balance": "none", "subset_size": None,
    "val_fraction": None, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": False, "keep_claims": True,
    "exclude_ids_path": None, "seed": 0,
}).to_pandas()

sample = pd.concat([
    te[te["class"] == cls].sample(n=n, random_state=SEED) for cls, n in SAMPLE_PER_CLASS.items()
]).reset_index(drop=True)

print(f"test : {len(te)} reviews | échantillon : {len(sample)}")
print(sample["class"].value_counts().to_dict())
print(f"longueur médiane (caractères) : {sample['content'].str.len().median():.0f} | max : {sample['content'].str.len().max()}")

INFO:src.data.dataset_semdetect:📦 Loading Sem-Detect dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_deb985b4e19d6e357800ffa800e5bf14.pt


test : 4410 reviews | échantillon : 100
{'human': 34, 'rewrite': 33, 'ai': 33}
longueur médiane (caractères) : 3416 | max : 11890


In [3]:
# notebook : semdetect_claims_local.ipynb — cellule 2 (remplace la précédente) : variantes et utilitaires
# variante -> (modèle servi, paramètres de LocalClaimExtractor) ; les paramètres entrent dans la clé de cache
VARIANTS = {
    "qwen3-8b": ("qwen3-8b", {}),
    "qwen3-14b": ("qwen3-14b", {}),
    # réflexion : les tokens de raisonnement comptent dans max_tokens -> marge relevée
    "qwen3-14b-think": ("qwen3-14b", {"enable_thinking": True, "max_tokens": 12000}),
    "qwen3-30b-a3b": ("qwen3-30b-a3b", {}),
    "qwen3-32b": ("qwen3-32b", {}),
}
TIMINGS_PATH = CACHE_ROOT / "claims_llm" / "timings_sample.json"


def served_models() -> list[str]:
    try:
        return [m.id for m in OpenAI(base_url=VLLM_URL, api_key="EMPTY", timeout=10).models.list().data]
    except Exception as e:
        print(f"serveur injoignable : {type(e).__name__}: {e}")
        return []


def make_extractor(variant: str, max_workers: int = 32) -> LocalClaimExtractor:
    served, kwargs = VARIANTS[variant]
    return LocalClaimExtractor(cache_root=CACHE_ROOT, served_model=served, base_url=VLLM_URL,
                               max_workers=max_workers, **kwargs)


def load_cache_records(variant: str) -> pd.DataFrame:
    """Enregistrements du cache d'une variante (aucun appel au LLM), indexés par review-id."""
    path = make_extractor(variant).cache_path
    if not path.exists():
        return pd.DataFrame()
    recs = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            recs[rec["review-id"]] = rec  # la dernière tentative l'emporte
    return pd.DataFrame(list(recs.values())).set_index("review-id")


def run_extraction(variant: str, df: pd.DataFrame, max_workers: int = 32) -> pd.DataFrame:
    served, _ = VARIANTS[variant]
    available = served_models()
    if served not in available:
        raise RuntimeError(f"le serveur sert {available}, pas {served!r} : rien n'est lancé, rien n'est mis en cache")

    before = load_cache_records(variant)
    done = set(before.index[before["status"] == "ok"]) if not before.empty else set()
    n_todo = int((~df["review-id"].isin(done)).sum())

    extractor = make_extractor(variant, max_workers)
    t0 = time.perf_counter()
    out = extractor.extract(df)
    elapsed = time.perf_counter() - t0
    print(f"{variant} : {elapsed:.0f} s pour {n_todo} reviews à extraire | source = {extractor.source_tag}")
    print(f"statuts : {out['claims_status'].value_counts().to_dict()}")

    if n_todo == len(df):  # on ne consigne que les passes complètes (pas les reprises partielles)
        timings = json.loads(TIMINGS_PATH.read_text()) if TIMINGS_PATH.exists() else {}
        timings[variant] = {"n": n_todo, "seconds": round(elapsed, 1), "source": extractor.source_tag}
        TIMINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
        TIMINGS_PATH.write_text(json.dumps(timings, indent=2), encoding="utf-8")
    return out


def claim_counts(raw: str) -> dict:
    c = parse_claims(raw)
    d = {cat: len(getattr(c, cat)) for cat in CLAIM_CATEGORIES}
    d["semantic"] = sum(d[cat] for cat in SEMANTIC_CATEGORIES)
    return d


print("modèles servis :", served_models())

modèles servis : ['qwen3-14b']


In [5]:
# notebook : semdetect_claims_local.ipynb — cellule 3 (remplace la précédente) : campagne automatique
import logging

from src.utils.vllm_manager import VLLMManager

logging.basicConfig(level=logging.INFO)

# modèle servi -> profil vLLM (SERVED_NAME du profil doit correspondre)
PROFILE_OF = {
    "qwen3-8b": "qwen3-8b-batch",
    "qwen3-14b": "qwen3-14b-batch",
    "qwen3-30b-a3b": "qwen3-30b-a3b-batch",
    "qwen3-32b": "qwen3-32b-batch",
}
CAMPAIGN = ["qwen3-14b-think", "qwen3-30b-a3b", "qwen3-32b"]  # ordre = ordre des serveurs

mgr = VLLMManager(gpu=1)
status = {}
for variant in CAMPAIGN:
    served, _ = VARIANTS[variant]
    profile = PROFILE_OF[served]
    try:
        if mgr.profile_served_name(profile) != served:
            raise ValueError(f"SERVED_NAME du profil {profile!r} != {served!r}")
        mgr.ensure(profile)
        run_extraction(variant, sample)
        status[variant] = "ok"
    except Exception as e:
        status[variant] = f"échec : {type(e).__name__}: {e}"
        print(f"\n!!! {variant} : {status[variant]}\n")

print("\nBilan de la campagne :")
for variant, s in status.items():
    print(f"  {variant:<18} {s[:300]}")

INFO:src.utils.vllm_manager:vLLM arrêté, GPU 1 libéré (18 MiB utilisés)
INFO:src.utils.vllm_manager:démarrage de qwen3-14b-batch (qwen3-14b) sur GPU 1, journal : /data2/cyaiche/vllm-env/logs/qwen3-14b-batch_20261003_110503.log
INFO:src.utils.vllm_manager:qwen3-14b prêt en 42 s
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-14b_6e0298dec8.jsonl : 99 ok
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-14b_6e0298dec8.jsonl : 99 ok
Claims (qwen3-14b): 100%|██████████| 1/1 [00:23<00:00, 23.68s/it]


qwen3-14b-think : 24 s pour 1 reviews à extraire | source = vllm:qwen3-14b:6e0298dec8
statuts : {'ok': 100}


INFO:src.utils.vllm_manager:vLLM arrêté, GPU 1 libéré (18 MiB utilisés)
INFO:src.utils.vllm_manager:démarrage de qwen3-30b-a3b-batch (qwen3-30b-a3b) sur GPU 1, journal : /data2/cyaiche/vllm-env/logs/qwen3-30b-a3b-batch_20261003_110614.log
INFO:src.utils.vllm_manager:qwen3-30b-a3b prêt en 52 s
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-30b-a3b_3de6babb6a.jsonl : 99 ok
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-30b-a3b_3de6babb6a.jsonl : 99 ok
Claims (qwen3-30b-a3b): 100%|██████████| 1/1 [01:04<00:00, 64.14s/it]


qwen3-30b-a3b : 64 s pour 1 reviews à extraire | source = vllm:qwen3-30b-a3b:3de6babb6a
statuts : {'ok': 99, 'error': 1}


INFO:src.utils.vllm_manager:vLLM arrêté, GPU 1 libéré (18 MiB utilisés)
INFO:src.utils.vllm_manager:démarrage de qwen3-32b-batch (qwen3-32b) sur GPU 1, journal : /data2/cyaiche/vllm-env/logs/qwen3-32b-batch_20261003_110815.log
INFO:src.utils.vllm_manager:qwen3-32b prêt en 252 s
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 0 ok
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 0 ok
Claims (qwen3-32b): 100%|██████████| 100/100 [03:05<00:00,  1.86s/it]

qwen3-32b : 186 s pour 100 reviews à extraire | source = vllm:qwen3-32b:0902bb5d60
statuts : {'ok': 100}

Bilan de la campagne :
  qwen3-14b-think    ok
  qwen3-30b-a3b      ok
  qwen3-32b          ok


In [ ]:
# notebook : semdetect_claims_local.ipynb — cellule 4 : tableau comparatif (lecture des caches uniquement)
sample_ids = list(sample["review-id"])
cls_all = sample.set_index("review-id")["class"]
timings = json.loads(TIMINGS_PATH.read_text(encoding="utf-8")) if TIMINGS_PATH.exists() else {}

claims_by_variant = {"gemini": sample.set_index("review-id")["claim_extraction"]}
stats_by_variant = {}
errors_by_variant = {}
for v in VARIANTS:
    rec = load_cache_records(v)
    if rec.empty:
        print(f"{v} : pas encore extrait, ignoré")
        continue
    rec = rec[rec.index.isin(sample_ids)]
    ok = rec[rec["status"] == "ok"]
    err = rec[rec["status"] != "ok"]
    claims_by_variant[v] = ok["claims"].map(lambda c: json.dumps(c, ensure_ascii=False))
    stats_by_variant[v] = {
        "n_ok": len(ok),
        "erreurs": len(err),
        "tokens_générés_moy": float(pd.to_numeric(ok.get("completion_tokens"), errors="coerce").mean()),
    }
    if not err.empty:
        errors_by_variant[v] = err.get("error", pd.Series(dtype=object)).fillna("(message absent)")

common = [rid for rid in sample_ids if all(rid in s.index for s in claims_by_variant.values())]
cls = cls_all.loc[common]
table = {name: pd.DataFrame([claim_counts(s.loc[rid]) for rid in common], index=common)
         for name, s in claims_by_variant.items()}
gem = table["gemini"]

rows = []
for name, t in table.items():
    row = {
        "variante": name,
        "sémantiques": t["semantic"].mean(),
        "evaluation": t["evaluation"].mean(),
        "constructive": t["constructiveInput"].mean(),
        "clarification": t["clarificationDialogue"].mean(),
        "< 4 sém.": (t["semantic"] < 4).mean(),
    }
    if name != "gemini":
        row["r vs Gemini"] = np.corrcoef(gem["semantic"], t["semantic"])[0, 1]
        row["ratio médian"] = (t["semantic"] / gem["semantic"].clip(lower=1)).median()
        row["tokens générés"] = stats_by_variant[name]["tokens_générés_moy"]
        row["erreurs"] = stats_by_variant[name]["erreurs"]
        t_info = timings.get(name)
        row["s / review"] = t_info["seconds"] / t_info["n"] if t_info else np.nan
    rows.append(row)

print(f"{len(common)} reviews communes à toutes les variantes (les reviews en erreur en sont exclues)\n")
print(pd.DataFrame(rows).set_index("variante").round(3).to_string(), "\n")

print("Claims sémantiques moyens par classe :")
print(pd.DataFrame({name: t["semantic"].groupby(cls).mean() for name, t in table.items()}).round(1), "\n")

if errors_by_variant:
    print("Détail des erreurs :")
    for v, msgs in errors_by_variant.items():
        for rid, msg in msgs.items():
            print(f"  [{v}] {rid} ({cls_all.get(rid, '?')}) : {str(msg)[:300]}")

In [7]:
# notebook : semdetect_claims_local.ipynb — cellule 5 : exemples côte à côte (catégorie evaluation)
HARD_HUMAN = "NeurIPS-2021-x2rdRAx3QF-0"  # 13 claims d'évaluation chez Gemini, 2 chez le 8B
examples = [HARD_HUMAN] + [next(r for r in common if cls.loc[r] == c and r != HARD_HUMAN) for c in ("rewrite", "ai")]

for rid in examples:
    print(f"\n=== {rid} ({cls_all.get(rid, '?')}) : evaluation ===")
    for name, s in claims_by_variant.items():
        if rid not in s.index:
            print(f"\n[{name}] absent (erreur d'extraction)")
            continue
        claims = parse_claims(s.loc[rid])
        print(f"\n[{name}] {len(claims.evaluation)} claims")
        for claim in claims.evaluation[:3]:
            print(f"  - {claim}")


=== NeurIPS-2021-x2rdRAx3QF-0 (human) : evaluation ===

[gemini] 13 claims
  - The paper presents a novel self-consistency objective with a thorough empirical analysis of its variants.
  - The unbiased analysis of the self-consistency objective's variants is an important contribution, even if not all experiments show a clear-cut benefit in all cases.
  - The proposed self-consistency update offers a new and important take on model updates in model-based RL, distinct from most prior work.

[qwen3-8b] 2 claims
  - Overall the paper in its current form is a significant and relevant contribution for the model-based RL community.
  - While not all experiments show a clear cut benefit of using self-consistency in all the cases visited in this paper, the unbiased analysis itself is an important contribution.

[qwen3-14b] 7 claims
  - The paper presents a novel self-consistency objective with a thorough empirical analysis, making it a significant and relevant contribution to the model-based R